In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

import re
import numpy as np
import pandas as pd


from inDecay import PATH,my_utils
os.chdir(PATH.main_dir)
from inDecay.models import Topk_Event_Overlapping

In [2]:
from tqdm.auto import tqdm
import pickle as pkl
from scipy.stats import pearsonr

pj = os.path.join

In [3]:
from qrguide import analysis_fn, transformation

In [4]:
## read Test set 
experiments =["ST_June_2017_BOB_LV7A_DPI7", "ST_June_2017_BOB_LV7B_DPI7" ,
      "ST_June_2017_CHO_LV7A_DPI7", "ST_June_2017_CHO_LV7B_DPI7",
      "ST_June_2017_E14TG2A_LV7A_DPI7", "ST_June_2017_E14TG2A_LV7B_DPI7",
      "ST_June_2017_HAP1_LV7A_DPI7", "ST_June_2017_HAP1_LV7B_DPI7",
      "ST_June_2017_K562_800x_LV7A_DPI7", "ST_June_2017_K562_800x_LV7B_DPI7",]

celltypes = [exp.split("_")[3] for exp in experiments if 'LV7A' in exp]
celltype_rename = ['iPSC', 'CHO', 'mESC', 'HAP1', 'K562']

In [6]:
a=[1,2,4,5,3]
b=[2,1,3,4,6]
np.intersect1d(a,b)

array([1, 2, 3, 4])

In [5]:
celltypes, celltype_rename

(['BOB', 'CHO', 'E14TG2A', 'HAP1', 'K562'],
 ['iPSC', 'CHO', 'mESC', 'HAP1', 'K562'])

In [6]:
pj = os.path.join

def read_pkl(path):
    with open(path, 'rb') as f:
        Y = pkl.load(f)
    f.close()
    return Y

def evalute_fn(Y_true_path, Y_pred_path):
    Y_pred =read_pkl(Y_pred_path)
    Y = read_pkl(Y_true_path)

    eval_json = analysis_fn.assessment_recipe_forecast(Y, Y_pred)
    eval_json.update(analysis_fn.assessment_recipe_IDL_forecast(Y, Y_pred))

    eval_df = pd.json_normalize(eval_json)
    return eval_df

def get_ratios(Y_true_path, Y_pred_path):

    pred_lookup =read_pkl(Y_pred_path)
    Y_lookup = read_pkl(Y_true_path)


    ratio_json = []
    for oligo, Y in Y_lookup.items():
        Y = Y.T

        pred = pred_lookup[oligo]
        Indel = Y[[0],:]
        y = Y[[1],:].astype("float32")

        # frameshift
        y_fs, pred_fs = analysis_fn.forecast_frameshift(y, pred, Indel)
        y_dr, pred_dr = analysis_fn.forecast_delratio(y, pred, Indel)

        ratio_json.append(
            {'Gene':oligo, "Rep1_frameshift":y_fs, "Pred_frameshift":pred_fs, "Rep_delratio":y_dr, "Pred_delratio":pred_dr}
        )
    
    return pd.json_normalize(ratio_json)

def ratio_error(row, ratio):
    error = row[f"Rep1_{ratio}"] - row[f'Pred_{ratio}']
    return np.abs(error).item()

# All_df['Abs. Err. Frameshift Ratio'] = All_df.apply(ratio_error , axis=1, args=('frameshift',))
# All_df['Abs. Err. Deletion Ratio'] = All_df.apply(ratio_error , axis=1, args=('delratio',))

# Key function for auto evalution

In [24]:
def find_pkl_and_eval(data_archive):
    """
    given the data_archive name, auto find 
    """
    archive_folder = f"{PATH.main_dir}/pl_trainer_log/{data_archive}_featv5_ST_DeepDecay_identity_C1"


    Cells = ['mESC']#['iPSC', 'mESC']
    Fix_settings = ['None', 'del_regressor[:1]', 'del_regressor[:2]']

    # 5 fold validation

    perform = []
    ratio_df = []

    for cell in Cells:
        for fix in Fix_settings:
            # for k_index in tqdm(range(5), desc=f"{cell} {fix}", leave=False):
            for k_index in range(0, 30):

                def annotate_df(df):
                    df['kfold_index'] = k_index
                    df['celltype'] = cell
                    df['fix_setting'] = fix
                
                # the directory name
                second_save_path = f"{cell}_featv5_c100_{fix}_30fold_{k_index}"

                # auto find pkl files
                Y_true = pj(archive_folder, second_save_path, "ForeCast_TestY.pkl")
                Y_baseline = pj(archive_folder, second_save_path, "Pretrained_Baseline_TestPred.pkl")
                Y_pred_path = my_utils.find_ckpt(pj(archive_folder, second_save_path, "lightning_logs"))
                Y_pred_path = Y_pred_path.replace(".ckpt", "TestPred.pkl")


                # get all the evaluation metrics            
                df_k = evalute_fn(Y_true, Y_pred_path)
                df_baseline_k = evalute_fn(Y_true, Y_baseline)
                df_baseline_k['celltype'] = cell

                # get frameshift / del ratio for each genes
                ratio_df_k = get_ratios(Y_true, Y_pred_path)
                ratio_df_baseline_k = get_ratios(Y_true, Y_baseline)
                ratio_df_baseline_k['celltype'] = cell

                ratio_df_k['Baseline_frameshift'] = ratio_df_baseline_k['Pred_frameshift']
                ratio_df_k['Baseline_delratio'] = ratio_df_baseline_k['Pred_delratio']

                # annotate
                for df in [df_k,df_baseline_k,ratio_df_k]:
                    annotate_df(df)

                df_baseline_k['fix_setting'] = 'baseline'

                perform.append(df_k)
                perform.append(df_baseline_k)
                ratio_df.append(ratio_df_k)
            
    perform  = pd.concat(perform)
    ratio_df = pd.concat(ratio_df)
    

    return perform, ratio_df 

# evaluate a single data archive

In [8]:
# use os.walk to return directories only
Sanger_archive_ls = [folder.split("_featv5")[0] for folder in os.listdir(PATH.pth_dir) if folder.startswith('Sanger')]
Sanger_archive_ls[:3], Sanger_archive_ls[-3:]

(['Sanger'], ['Sanger'])

## find Ytrue and Ypred pkl

In [9]:
from tqdm.auto import tqdm

In [25]:
# which takes around 14 seconds
perform, ratio_df = find_pkl_and_eval("Sanger")

perform_mean = perform.groupby(["celltype", "fix_setting"]).agg("mean").reset_index()
# perform_baseline = perform_baseline.groupby(["celltype"]).agg("mean")

In [26]:
perform_mean

,celltype,fix_setting,KL divergence,Top1 events recall,Top2 events recall,Top3 events recall,Top5 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top1_IDL,Top2_IDL,Top3_IDL,Top5_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,kfold_index
0,mESC,None,5.747540,0.533333,1.100000,1.433333,2.433333,0.0,2.800000,4.366667,0.902488,0.566667,1.200000,1.800000,3.066667,0.197926,0.0,0.434195,14.5
1,mESC,baseline,5.613203,0.666667,1.133333,1.533333,2.033333,0.0,2.433333,4.100000,0.863357,0.733333,1.266667,1.733333,2.866667,0.185147,0.0,0.430992,14.5
2,mESC,del_regressor[:1],5.648402,0.500000,1.033333,1.433333,2.433333,0.0,2.666667,4.333333,0.906042,0.533333,1.200000,1.733333,3.000000,0.196036,0.0,0.427332,14.5
3,mESC,del_regressor[:2],5.870309,0.466667,1.033333,1.433333,2.433333,0.0,2.800000,4.466667,0.912404,0.466667,1.166667,1.666667,2.966667,0.193287,0.0,0.428299,14.5


In [20]:
perform_mean

,celltype,fix_setting,KL divergence,Top1 events recall,Top2 events recall,Top3 events recall,Top5 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top1_IDL,Top2_IDL,Top3_IDL,Top5_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,kfold_index
0,mESC,None,5.388283,0.565217,1.130435,1.434783,2.565217,0.0,2.826087,4.260870,0.923609,0.652174,1.217391,1.695652,3.043478,0.191751,0.0,0.432440,12.0
1,mESC,baseline,5.362644,0.608696,1.086957,1.521739,2.086957,0.0,2.434783,3.956522,0.907395,0.695652,1.260870,1.739130,2.782609,0.189847,0.0,0.424693,12.0
2,mESC,del_regressor[:1],5.146473,0.521739,1.043478,1.521739,2.521739,0.0,2.652174,4.260870,0.896706,0.565217,1.173913,1.652174,2.913043,0.188625,0.0,0.425629,12.0
3,mESC,del_regressor[:2],5.462353,0.478261,1.043478,1.478261,2.478261,0.0,2.826087,4.391304,0.933397,0.521739,1.173913,1.565217,2.913043,0.186783,0.0,0.423362,12.0


In [17]:
perform_mean

,celltype,fix_setting,KL divergence,Top1 events recall,Top2 events recall,Top3 events recall,Top5 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top1_IDL,Top2_IDL,Top3_IDL,Top5_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,kfold_index
0,mESC,None,5.329737,0.590909,1.181818,1.454545,2.590909,0.0,2.863636,4.181818,0.931935,0.681818,1.272727,1.772727,3.045455,0.196739,0.0,0.426194,11.5
1,mESC,baseline,5.332157,0.636364,1.136364,1.590909,2.181818,0.0,2.545455,3.954545,0.924406,0.727273,1.318182,1.727273,2.818182,0.195656,0.0,0.416927,11.5
2,mESC,del_regressor[:1],5.095178,0.545455,1.090909,1.545455,2.545455,0.0,2.681818,4.181818,0.904679,0.590909,1.227273,1.727273,2.863636,0.193745,0.0,0.419511,11.5
3,mESC,del_regressor[:2],5.418984,0.500000,1.090909,1.500000,2.500000,0.0,2.863636,4.318182,0.940571,0.545455,1.227273,1.636364,2.954545,0.191502,0.0,0.416849,11.5


In [14]:
perform_mean

,celltype,fix_setting,KL divergence,Top1 events recall,Top2 events recall,Top3 events recall,Top5 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top1_IDL,Top2_IDL,Top3_IDL,Top5_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,kfold_index
0,mESC,None,5.496357,0.666667,1.200000,1.466667,2.733333,0.0,2.933333,4.133333,0.906455,0.733333,1.266667,1.800000,3.066667,0.234877,0.0,0.419503,8.0
1,mESC,baseline,5.115355,0.666667,1.133333,1.600000,2.133333,0.0,2.466667,4.000000,0.862822,0.733333,1.333333,1.733333,2.666667,0.224508,0.0,0.407326,8.0
2,mESC,del_regressor[:1],5.196524,0.600000,1.066667,1.600000,2.600000,0.0,2.666667,4.200000,0.867398,0.666667,1.200000,1.733333,2.800000,0.228764,0.0,0.411462,8.0
3,mESC,del_regressor[:2],5.568037,0.533333,1.133333,1.666667,2.600000,0.0,2.866667,4.400000,0.891506,0.600000,1.266667,1.666667,2.933333,0.223647,0.0,0.413428,8.0


In [ ]:
perform_mean

,celltype,fix_setting,KL divergence,Top1 events recall,Top2 events recall,Top3 events recall,Top5 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top1_IDL,Top2_IDL,Top3_IDL,Top5_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,kfold_index
0,mESC,None,5.812062,0.517241,1.103448,1.482759,2.310345,0.0,2.724138,4.448276,0.953299,0.620690,1.172414,1.655172,3.000000,0.201858,0.0,0.427141,15.0
1,mESC,baseline,5.645115,0.655172,1.103448,1.517241,2.034483,0.0,2.379310,4.034483,0.886458,0.689655,1.241379,1.689655,2.896552,0.192058,0.0,0.430901,15.0
2,mESC,del_regressor[:1],5.713867,0.517241,1.068966,1.482759,2.275862,0.0,2.620690,4.413793,0.980515,0.586207,1.172414,1.655172,3.000000,0.193865,0.0,0.430011,15.0
3,mESC,del_regressor[:2],5.589877,0.586207,1.137931,1.517241,2.413793,0.0,2.724138,4.413793,0.916781,0.620690,1.206897,1.793103,3.103448,0.193618,0.0,0.437959,15.0


In [23]:
perform_mean

,celltype,fix_setting,KL divergence,Top1 events recall,Top2 events recall,Top3 events recall,Top5 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top1_IDL,Top2_IDL,Top3_IDL,Top5_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,kfold_index
0,mESC,None,5.898149,0.500000,1.033333,1.433333,2.300000,0.0,2.633333,4.400000,0.920231,0.566667,1.200000,1.666667,2.966667,0.194734,0.0,0.427157,14.5
1,mESC,baseline,5.737073,0.666667,1.100000,1.500000,2.033333,0.0,2.400000,4.033333,0.884155,0.700000,1.233333,1.666667,2.900000,0.189093,0.0,0.429039,14.5
2,mESC,del_regressor[:1],5.579078,0.500000,1.133333,1.666667,2.400000,0.0,2.766667,4.300000,0.932458,0.500000,1.233333,1.766667,3.100000,0.197482,0.0,0.429083,14.5
3,mESC,del_regressor[:2],5.949835,0.500000,1.066667,1.533333,2.233333,0.0,2.700000,4.333333,0.977438,0.500000,1.133333,1.733333,3.033333,0.203511,0.0,0.429118,14.5


In [13]:
ratio_df.head()

,Gene,Rep1_frameshift,Pred_frameshift,Rep_delratio,Pred_delratio,Baseline_frameshift,Baseline_delratio,kfold_index,celltype,fix_setting
0,H2K1-sg2,0.795224,0.811379,0.638206,0.700373,0.822357,0.924117,0,iPSC,None
1,Rag1-sg1-4,0.888889,0.930068,0.888889,0.295772,0.897242,0.875335,0,iPSC,None
2,sry,1.000000,0.857497,1.000000,0.851633,0.911124,0.968655,0,iPSC,None
3,ZNRF4-sg2,0.865988,0.946290,0.896298,0.304272,0.860473,0.716432,0,iPSC,None
4,TPO,0.926199,0.813741,0.734317,0.783664,0.650772,0.902255,0,iPSC,None


# iterate all data archive

In [11]:
perform_all = []
ratio_df_all = []

for data_archive in tqdm(Sanger_archive_ls, desc='iter data', leave=False):

    try:
        perform,  ratio_df = find_pkl_and_eval(data_archive)

        perform_mean = perform.groupby(["celltype", "fix_setting"]).agg("mean").reset_index()

        perform['data_archive'] = data_archive
        ratio_df['data_archive'] = data_archive

        
        perform_all.append(perform)
        ratio_df_all.append(ratio_df)
        
    except FileNotFoundError:
        print(data_archive, 'file missing')

perform_all = pd.concat(perform_all)
ratio_df_all = pd.concat(ratio_df_all)

iter data:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
perform_all.to_csv("results/Sanger_benchmark/mESC_C200_30fold_perform_Sep05.csv", index=False)

In [33]:
perform_all.data_archive.nunique()

4

In [13]:
perform_all.groupby(['data_archive','fix_setting']).mean(numeric_only=True).sort_values('Top2 events recall',ascending=False)

,,KL divergence,Top1 events recall,Top2 events recall,Top3 events recall,Top5 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top1_IDL,Top2_IDL,Top3_IDL,Top5_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,kfold_index
data_archive,fix_setting,,,,,,,,,,,,,,,,,
Sanger_decr2_invalidlabel_LowCount,None,5.771159,0.551724,1.206897,1.620690,2.413793,0.0,2.827586,4.413793,0.883974,0.655172,1.241379,1.793103,3.000000,0.221230,0.0,0.429230,15.0
Sanger_decr2_invalidlabel_NoICE,del_regressor[:2],5.883176,0.551724,1.206897,1.482759,2.448276,0.0,2.827586,4.379310,0.892144,0.586207,1.241379,1.724138,3.137931,0.212074,0.0,0.428976,15.0
Sanger_decr2_NoICE_LowCount,del_regressor[:2],5.837042,0.517241,1.172414,1.517241,2.413793,0.0,2.827586,4.448276,0.880698,0.586207,1.275862,1.724138,3.103448,0.223443,0.0,0.431815,15.0
Sanger_decr2_NoICE,None,5.611972,0.448276,1.172414,1.482759,2.275862,0.0,2.724138,4.482759,0.857841,0.551724,1.241379,1.827586,3.103448,0.199819,0.0,0.432929,15.0
Sanger_icecosine_invalidlabel,baseline,5.657645,0.655172,1.137931,1.517241,2.034483,0.0,2.448276,4.137931,0.869399,0.689655,1.241379,1.689655,2.862069,0.200933,0.0,0.431267,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Sanger_decr2_icecosine_invalidlabel,del_regressor[:1],5.955294,0.517241,1.000000,1.379310,2.448276,0.0,2.827586,4.482759,0.946322,0.551724,1.137931,1.689655,3.068966,0.209931,0.0,0.429120,15.0
Sanger_icecosine_NoICE,del_regressor[:2],5.902397,0.517241,1.000000,1.413793,2.344828,0.0,2.586207,4.482759,0.914676,0.551724,1.137931,1.758621,2.896552,0.194702,0.0,0.426925,15.0
Sanger_invalidlabel_NoICE,del_regressor[:2],5.754118,0.586207,1.000000,1.551724,2.310345,0.0,2.689655,4.344828,0.913284,0.517241,1.206897,1.724138,3.103448,0.200983,0.0,0.427829,15.0


In [34]:
perform_all.query("`data_archive`=='Sanger_decr2_invalidlabel_LowCount'").groupby(['celltype','fix_setting']).mean(numeric_only=True).sort_values(by=['Top5 events recall'],ascending= False)

,,KL divergence,Top1 events recall,Top2 events recall,Top3 events recall,Top5 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top1_IDL,Top2_IDL,Top3_IDL,Top5_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,kfold_index
celltype,fix_setting,,,,,,,,,,,,,,,,,


In [57]:
perform_all.query("`data_archive`=='Sanger_decr2_icecosine_LowCount'").groupby(['celltype','fix_setting']).mean(numeric_only=True).sort_values(by=['KL divergence'])

KL divergence  Top5 events recall  \
celltype fix_setting                                            
iPSC     None                    6.006075            2.352381   
mESC     baseline                6.080334            2.042857   
iPSC     del_regressor[:2]       6.171076            2.352381   
         del_regressor[:1]       6.230537            2.319048   
         baseline                6.240662            2.176190   
mESC     del_regressor[:2]       6.280984            2.419048   
         None                    6.384351            2.457143   
         del_regressor[:1]       6.459778            2.380952   

                            Top10 events recall  R2 of Frameshift ratio  \
celltype fix_setting                                                      
iPSC     None                          3.828571                0.216478   
mESC     baseline                      3.542857                0.276315   
iPSC     del_regressor[:2]             3.628571                0.117136   
         del_regressor[:1]             3.690476                0.209900   
         baseline                      3.604762                0.247724   
mESC     del_regressor[:2]             3.890476                0.130812   
         None                          3.819048                0.095986   
         del_regressor[:1]             3.957143                0.165055   

                            Coll_I_Top5  Coll_I_Top10   KLD_IDL  Top5_IDL  \
celltype fix_setting                                                        
iPSC     None                  2.538095      4.285714  1.055171  2.766667   
mESC     baseline              2.447619      4.000000  1.059970  2.842857   
iPSC     del_regressor[:2]     2.576190      4.085714  1.131740  2.738095   
         del_regressor[:1]     2.509524      4.247619  1.050078  2.738095   
         baseline              2.547619      4.223810  1.039384  2.742857   
mESC     del_regressor[:2]     2.614286      4.223810  1.004547  2.900000   
         None                  2.642857      4.219048  1.041011  2.966667   
         del_regressor[:1]     2.700000      4.161905  1.178852  2.861905   

                            Top10_IDL  W1-distance_IDL  delratio_r2  \
celltype fix_setting                                                  
iPSC     None                4.552381         0.231725     0.335150   
mESC     baseline            4.947619         0.230508     0.627035   
iPSC     del_regressor[:2]   4.619048         0.242849     0.296972   
         del_regressor[:1]   4.690476         0.219018     0.424952   
         baseline            5.038095         0.226305     0.642671   
mESC     del_regressor[:2]   4.719048         0.238203     0.456493   
         None                4.523810         0.247946     0.250364   
         del_regressor[:1]   4.390476         0.268024     0.217007   

                            Kendall_tau_IDL  kfold_index  
celltype fix_setting                                      
iPSC     None                      0.374385          2.0  
mESC     baseline                  0.386256          2.0  
iPSC     del_regressor[:2]         0.373761          2.0  
         del_regressor[:1]         0.383598          2.0  
         baseline                  0.389148          2.0  
mESC     del_regressor[:2]         0.387443          2.0  
         None                      0.377923          2.0  
         del_regressor[:1]         0.373643          2.0

In [16]:
perform_all.groupby(['celltype','data_archive']).mean(numeric_only=True).sort_values(by='Top5 events recall',ascending=False)

KL divergence  \
celltype data_archive                                                  
iPSC     Sanger_decr2_icecosine                             6.299978   
         Sanger_decr2_icecosine_LowCount                    6.188279   
         Sanger_decr2_icecosine_invalidlabel_LowCount       6.509671   
         Sanger_decr2                                       6.284958   
         Sanger_decr2_icecosine_invalidlabel                6.494061   
         Sanger_decr2_invalidlabel                          6.546266   
         Sanger_decr2_invalidlabel_LowCount                 6.278345   
mESC     Sanger_decr2_icecosine_LowCount                    6.227686   
         Sanger_decr2_invalidlabel_LowCount                 6.261303   
         Sanger_decr2_icecosine_invalidlabel                6.275204   
         Sanger_decr2_invalidlabel                          6.274209   
         Sanger_decr2_icecosine_invalidlabel_LowCount       6.155895   
iPSC     Sanger_icecosine_NoICE                             6.425709   
         Sanger_icecosine_invalidlabel_NoICE_LowCount       6.393731   
mESC     Sanger_decr2_icecosine                             6.223064   
         Sanger_decr2                                       6.267248   
         Sanger_icecosine_NoICE_LowCount                    6.247817   
iPSC     Sanger_icecosine_NoICE_LowCount                    6.452345   
mESC     Sanger_icecosine_invalidlabel_NoICE_LowCount       6.247978   
         Sanger_icecosine_NoICE                             6.245294   
iPSC     Sanger_NoICE                                       6.975780   
         Sanger_invalidlabel_NoICE_LowCount                 6.962693   
         Sanger_icecosine_LowCount                          6.783450   
         Sanger_NoICE_LowCount                              7.028006   
         Sanger_invalidlabel_LowCount                       7.033905   
         Sanger_icecosine                                   6.921884   
         Sanger_icecosine_invalidlabel                      6.955016   
mESC     Sanger_invalidlabel_NoICE_LowCount                 6.878612   
         Sanger_NoICE_LowCount                              6.787113   
         Sanger_NoICE                                       6.752613   
         Sanger_icecosine_LowCount                          6.791235   
         Sanger_icecosine                                   6.702719   
         Sanger_icecosine_invalidlabel                      6.755286   
         Sanger_invalidlabel_LowCount                       6.786082   

                                                       Top5 events recall  \
celltype data_archive                                                       
iPSC     Sanger_decr2_icecosine                                  2.269841   
         Sanger_decr2_icecosine_LowCount                         2.258730   
         Sanger_decr2_icecosine_invalidlabel_LowCount            2.258730   
         Sanger_decr2                                            2.254762   
         Sanger_decr2_icecosine_invalidlabel                     2.253175   
         Sanger_decr2_invalidlabel                               2.242857   
         Sanger_decr2_invalidlabel_LowCount                      2.242063   
mESC     Sanger_decr2_icecosine_LowCount                         2.230952   
         Sanger_decr2_invalidlabel_LowCount                      2.203175   
         Sanger_decr2_icecosine_invalidlabel                     2.200794   
         Sanger_decr2_invalidlabel                               2.198413   
         Sanger_decr2_icecosine_invalidlabel_LowCount            2.191270   
iPSC     Sanger_icecosine_NoICE                                  2.184921   
         Sanger_icecosine_invalidlabel_NoICE_LowCount            2.184921   
mESC     Sanger_decr2_icecosine                                  2.176984   
         Sanger_decr2                                            2.175397   
         Sanger_icecosine_NoICE_LowCount                         2.153968   

In [45]:
import collections
allcounts= perform_all.groupby(['data_archive']).mean(numeric_only=True).sort_values(by='Top5 events recall',ascending=False)[0:10].index
summ=[]
for i in allcounts:
    summ.extend(i.split('_')[0:])
summ=list(filter(lambda a: a!='Sanger', summ))
counter = collections.Counter(summ)
counter

Counter({'decr2': 7,
         'icecosine': 7,
         'LowCount': 5,
         'invalidlabel': 5,
         'NoICE': 3})

In [38]:
import collections
allcounts= perform_all.groupby(['data_archive']).mean(numeric_only=True).sort_values(by='KL divergence',ascending=True)[0:10].index
summ=[]
for i in allcounts:
    summ.extend(i.split('_')[0:])
summ=list(filter(lambda a: a!='Sanger', summ))
counter = collections.Counter(summ)
counter

Counter({'decr2': 7,
         'icecosine': 7,
         'LowCount': 5,
         'invalidlabel': 5,
         'NoICE': 3})

In [18]:
perform_all.groupby(['celltype','data_archive']).mean(numeric_only=True).sort_values(by='KL divergence',ascending=True)

KL divergence  \
celltype data_archive                                                  
mESC     Sanger_decr2_icecosine_invalidlabel_LowCount       6.155895   
iPSC     Sanger_decr2_icecosine_LowCount                    6.188279   
mESC     Sanger_decr2_icecosine                             6.223064   
         Sanger_decr2_icecosine_LowCount                    6.227686   
         Sanger_icecosine_NoICE                             6.245294   
         Sanger_icecosine_NoICE_LowCount                    6.247817   
         Sanger_icecosine_invalidlabel_NoICE_LowCount       6.247978   
         Sanger_decr2_invalidlabel_LowCount                 6.261303   
         Sanger_decr2                                       6.267248   
         Sanger_decr2_invalidlabel                          6.274209   
         Sanger_decr2_icecosine_invalidlabel                6.275204   
iPSC     Sanger_decr2_invalidlabel_LowCount                 6.278345   
         Sanger_decr2                                       6.284958   
         Sanger_decr2_icecosine                             6.299978   
         Sanger_icecosine_invalidlabel_NoICE_LowCount       6.393731   
         Sanger_icecosine_NoICE                             6.425709   
         Sanger_icecosine_NoICE_LowCount                    6.452345   
         Sanger_decr2_icecosine_invalidlabel                6.494061   
         Sanger_decr2_icecosine_invalidlabel_LowCount       6.509671   
         Sanger_decr2_invalidlabel                          6.546266   
mESC     Sanger_icecosine                                   6.702719   
         Sanger_NoICE                                       6.752613   
         Sanger_icecosine_invalidlabel                      6.755286   
iPSC     Sanger_icecosine_LowCount                          6.783450   
mESC     Sanger_invalidlabel_LowCount                       6.786082   
         Sanger_NoICE_LowCount                              6.787113   
         Sanger_icecosine_LowCount                          6.791235   
         Sanger_invalidlabel_NoICE_LowCount                 6.878612   
iPSC     Sanger_icecosine                                   6.921884   
         Sanger_icecosine_invalidlabel                      6.955016   
         Sanger_invalidlabel_NoICE_LowCount                 6.962693   
         Sanger_NoICE                                       6.975780   
         Sanger_NoICE_LowCount                              7.028006   
         Sanger_invalidlabel_LowCount                       7.033905   

                                                       Top5 events recall  \
celltype data_archive                                                       
mESC     Sanger_decr2_icecosine_invalidlabel_LowCount            2.191270   
iPSC     Sanger_decr2_icecosine_LowCount                         2.258730   
mESC     Sanger_decr2_icecosine                                  2.176984   
         Sanger_decr2_icecosine_LowCount                         2.230952   
         Sanger_icecosine_NoICE                                  2.124603   
         Sanger_icecosine_NoICE_LowCount                         2.153968   
         Sanger_icecosine_invalidlabel_NoICE_LowCount            2.130952   
         Sanger_decr2_invalidlabel_LowCount                      2.203175   
         Sanger_decr2                                            2.175397   
         Sanger_decr2_invalidlabel                               2.198413   
         Sanger_decr2_icecosine_invalidlabel                     2.200794   
iPSC     Sanger_decr2_invalidlabel_LowCount                      2.242063   
         Sanger_decr2                                            2.254762   
         Sanger_decr2_icecosine                                  2.269841   
         Sanger_icecosine_invalidlabel_NoICE_LowCount            2.184921   
         Sanger_icecosine_NoICE                                  2.184921   
         Sanger_icecosine_NoICE_LowCount                         2.139683   

In [47]:
perform_all.groupby(['celltype','fix_setting','data_archive']).agg('mean').sort_values(by='Top5 events recall',ascending=False)[0:10]

KL divergence  \
celltype fix_setting       data_archive                                         
mESC     None              Sanger_decr2_icecosine_LowCount           6.384351   
         del_regressor[:2] Sanger_decr2_icecosine_invalidlabel       6.630549   
                           Sanger_decr2_icecosine_LowCount           6.280984   
         None              Sanger_decr2_icecosine                    6.377719   
         del_regressor[:1] Sanger_decr2_invalidlabel                 6.442816   
iPSC     del_regressor[:2] Sanger_decr2_icecosine                    6.283939   
mESC     del_regressor[:1] Sanger_decr2_icecosine_invalidlabel       6.436612   
                           Sanger_decr2_invalidlabel_LowCount        6.457106   
iPSC     del_regressor[:1] Sanger_decr2_invalidlabel                 6.612640   
mESC     del_regressor[:1] Sanger_decr2_icecosine_LowCount           6.459778   

                                                                Top5 events recall  \
celltype fix_setting       data_archive                                              
mESC     None              Sanger_decr2_icecosine_LowCount                2.457143   
         del_regressor[:2] Sanger_decr2_icecosine_invalidlabel            2.423810   
                           Sanger_decr2_icecosine_LowCount                2.419048   
         None              Sanger_decr2_icecosine                         2.419048   
         del_regressor[:1] Sanger_decr2_invalidlabel                      2.419048   
iPSC     del_regressor[:2] Sanger_decr2_icecosine                         2.385714   
mESC     del_regressor[:1] Sanger_decr2_icecosine_invalidlabel            2.385714   
                           Sanger_decr2_invalidlabel_LowCount             2.385714   
iPSC     del_regressor[:1] Sanger_decr2_invalidlabel                      2.385714   
mESC     del_regressor[:1] Sanger_decr2_icecosine_LowCount                2.380952   

                                                                Top10 events recall  \
celltype fix_setting       data_archive                                               
mESC     None              Sanger_decr2_icecosine_LowCount                 3.819048   
         del_regressor[:2] Sanger_decr2_icecosine_invalidlabel             3.957143   
                           Sanger_decr2_icecosine_LowCount                 3.890476   
         None              Sanger_decr2_icecosine                          3.890476   
         del_regressor[:1] Sanger_decr2_invalidlabel                       4.023810   
iPSC     del_regressor[:2] Sanger_decr2_icecosine                          3.890476   
mESC     del_regressor[:1] Sanger_decr2_icecosine_invalidlabel             3.919048   
                           Sanger_decr2_invalidlabel_LowCount              3.952381   
iPSC     del_regressor[:1] Sanger_decr2_invalidlabel                       3.766667   
mESC     del_regressor[:1] Sanger_decr2_icecosine_LowCount                 3.957143   

                                                                R2 of Frameshift ratio  \
celltype fix_setting       data_archive                                                  
mESC     None              Sanger_decr2_icecosine_LowCount                    0.095986   
         del_regressor[:2] Sanger_decr2_icecosine_invalidlabel                0.145609   
                           Sanger_decr2_icecosine_LowCount                    0.130812   
         None              Sanger_decr2_icecosine                             0.199677   
         del_regressor[:1] Sanger_decr2_invalidlabel                          0.178175   
iPSC     del_regressor[:2] Sanger_decr2_icecosine                             0.165919   
mESC     del_regressor[:1] Sanger_decr2_icecosine_invalidlabel                0.190224   
                           Sanger_decr2_invalidlabel_LowCount                 0.187565   
iPSC     del_regressor[:1] Sanger_decr2_invalidlabel                          0.114631   
mESC     del_regressor[

In [49]:
perform_all.groupby(['celltype','fix_setting','data_archive']).agg('mean').sort_values(by='KL divergence',ascending=True)[0:10]

KL divergence  \
celltype fix_setting       data_archive                                                  
iPSC     None              Sanger_decr2_icecosine_LowCount                    6.006075   
mESC     baseline          Sanger_decr2_invalidlabel                          6.070827   
                           Sanger_decr2                                       6.070827   
                           Sanger_decr2_icecosine_invalidlabel                6.071066   
                           Sanger_decr2_icecosine                             6.071066   
                           Sanger_decr2_invalidlabel_LowCount                 6.080096   
                           Sanger_decr2_icecosine_invalidlabel_LowCount       6.080334   
                           Sanger_decr2_icecosine_LowCount                    6.080334   
iPSC     None              Sanger_decr2_invalidlabel_LowCount                 6.102840   
mESC     del_regressor[:1] Sanger_decr2_icecosine_invalidlabel_LowCount       6.112305   

                                                                         Top5 events recall  \
celltype fix_setting       data_archive                                                       
iPSC     None              Sanger_decr2_icecosine_LowCount                         2.352381   
mESC     baseline          Sanger_decr2_invalidlabel                               2.042857   
                           Sanger_decr2                                            2.042857   
                           Sanger_decr2_icecosine_invalidlabel                     2.042857   
                           Sanger_decr2_icecosine                                  2.042857   
                           Sanger_decr2_invalidlabel_LowCount                      2.042857   
                           Sanger_decr2_icecosine_invalidlabel_LowCount            2.042857   
                           Sanger_decr2_icecosine_LowCount                         2.042857   
iPSC     None              Sanger_decr2_invalidlabel_LowCount                      2.319048   
mESC     del_regressor[:1] Sanger_decr2_icecosine_invalidlabel_LowCount            2.328571   

                                                                         Top10 events recall  \
celltype fix_setting       data_archive                                                        
iPSC     None              Sanger_decr2_icecosine_LowCount                          3.828571   
mESC     baseline          Sanger_decr2_invalidlabel                                3.542857   
                           Sanger_decr2                                             3.542857   
                           Sanger_decr2_icecosine_invalidlabel                      3.542857   
                           Sanger_decr2_icecosine                                   3.542857   
                           Sanger_decr2_invalidlabel_LowCount                       3.542857   
                           Sanger_decr2_icecosine_invalidlabel_LowCount             3.542857   
                           Sanger_decr2_icecosine_LowCount                          3.542857   
iPSC     None              Sanger_decr2_invalidlabel_LowCount                       3.828571   
mESC     del_regressor[:1] Sanger_decr2_icecosine_invalidlabel_LowCount             4.157143   

                                                                         R2 of Frameshift ratio  \
celltype fix_setting       data_archive                                                           
iPSC     None              Sanger_decr2_icecosine_LowCount                             0.216478   
mESC     baseline          Sanger_decr2_invalidlabel                                   0.275591   
                           Sanger_decr2                                                0.275591   
                           Sanger_decr2_icecosine_invalidlabel                         0.276066   
                           Sanger_decr2_icecosine                                      0.276066   
     

In [ ]:
perform_all.groupby(['celltype','fix_setting','data_archive']).agg('mean').sort_values(by='KL divergence',ascending=True)[0:10]

In [82]:
print(perform_all.shape)
perform_all.to_csv("results/Sanger_benchmark/17archive_36gens_5fold_perform_Aug29.csv", index=False)

(1020, 16)


In [ ]:
print(ratio_df_all.shape)
ratio_df_all.to_csv("results/Sanger_benchmark/17archive_36gens_5fold_fsdel_ratio_Aug29.csv", index=False)

(3372, 11)


#### mESC

In [98]:
perform_all.query("`fix_setting`!='baseline' & `celltype` == 'mESC'").max(axis=0)[:-2]

KL divergence             10.405339
Top5 events recall              3.0
Top10 events recall        4.833333
R2 of Frameshift ratio     0.952038
Coll_I_Top5                     3.5
Coll_I_Top10               5.285714
KLD_IDL                    1.998706
Top5_IDL                   3.833333
Top10_IDL                       6.0
W1-distance_IDL            0.397284
delratio_r2                0.888743
Kendall_tau_IDL            0.468327
kfold_index                       4
celltype                       mESC
dtype: object

In [99]:
perform_all.query("`fix_setting`=='baseline' & `celltype` == 'mESC' ").max(axis=0)[:-2]

KL divergence             8.774204
Top5 events recall             2.5
Top10 events recall       4.428571
R2 of Frameshift ratio    0.959041
Coll_I_Top5                    3.0
Coll_I_Top10              4.857143
KLD_IDL                   1.946787
Top5_IDL                  3.333333
Top10_IDL                 6.166667
W1-distance_IDL           0.362196
delratio_r2               0.906837
Kendall_tau_IDL           0.483522
kfold_index                      4
celltype                      mESC
dtype: object

#### iPSC

In [97]:
perform_all.query("`fix_setting`!='baseline' & `celltype` == 'iPSC'").max(axis=0)[:-2]

KL divergence             10.404295
Top5 events recall         2.833333
Top10 events recall        4.833333
R2 of Frameshift ratio     0.864038
Coll_I_Top5                3.166667
Coll_I_Top10                    5.0
KLD_IDL                    2.186483
Top5_IDL                   3.666667
Top10_IDL                  6.333333
W1-distance_IDL            0.363494
delratio_r2                0.863577
Kendall_tau_IDL            0.473748
kfold_index                       4
celltype                       iPSC
dtype: object

In [ ]:
perform_all.query("`fix_setting`=='baseline' & `celltype` == 'iPSC' ").max(axis=0)

KL divergence                                       8.919365
Top5 events recall                                  2.666667
Top10 events recall                                 4.428571
R2 of Frameshift ratio                              0.959041
Coll_I_Top5                                              3.0
Coll_I_Top10                                        5.142857
KLD_IDL                                             1.960333
Top5_IDL                                                 3.5
Top10_IDL                                           6.333333
W1-distance_IDL                                     0.362196
delratio_r2                                         0.928965
Kendall_tau_IDL                                     0.485436
kfold_index                                                4
celltype                                                mESC
fix_setting                                         baseline
data_archive              Sanger_invalidlabel_NoICE_LowCount
dtype: object